# Практика по S2T и T2S
В этой тетрадке выполняем практические задания: распознавание речи, построение спектрограмм, сравнение моделей, подсчёт WER, синтез речи, а также самостоятельные упражнения.

## 1. Запуск готовой S2T модели
Распознаем текст из аудиофайла с помощью Whisper (нейросеть от OpenAI для распознавания речи).

In [ ]:
!pip install -U openai-whisper
import whisper
model = whisper.load_model("small")
result = model.transcribe("audio.mp3", language="ru")
print(result["text"])

## 2. Построение спектрограммы

Напоминалка:

Спектрограмма — изображение, показывающее, как распределяется энергия звука по частотам во времени.

Мел-спектрограмма — переведённая в мел-шкалу, т.е. шкалу восприятия частот человеком.

In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt

# Загрузка аудио
y, sr = librosa.load("audio.mp3", sr=16000)

# Расчет Mel-спектрограммы
S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)

# Построение
plt.figure(figsize=(10, 4))
librosa.display.specshow(librosa.power_to_db(S, ref=np.max),
                         sr=sr, x_axis='time', y_axis='mel')
plt.colorbar(format='%+2.0f dB')
plt.title('Mel-Spectrogram')
plt.tight_layout()
plt.show()

## 3. Сравнение моделей

Wav2Vec2.0 — модель от Facebook (Meta), работающая напрямую с сырой волной (raw audio).
В отличие от Whisper, она не использует мел-спектрограммы — только свёртки и трансформер.

Главное преимущество Whisper — очень высокая устойчивость к шумам и некачественным записям, чего нет у Wav2Vec2.

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import torch, librosa

# Загружаем процессор Wav2Vec2.
# Он отвечает за нормализацию аудио (масштабирование), преобразование в входные тензоры PyTorch
# и последующую расшифровку выходных ID в текст
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")

# Модель принимает аудиотензор и возвращает логиты (предсказания вероятностей символов).
model_w2v = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

# Загружаем аудио через librosa. sr=16000 — частота дискретизации для Wav2Vec2
audio, rate = librosa.load("audio.mp3", sr=16000)

# processor выполняет несколько действий:
# 1. нормализует аудиосигнал
# 2. превращает в батч нужной формы
# 3. конвертирует в PyTorch-тензор
input_values = processor(audio, sampling_rate=rate, return_tensors="pt", padding=True).input_values

# Отключаем градиенты, чтобы ускорить вычисления и уменьшить память.
# model_w2v делает прямой проход модели и возвращает логиты.
with torch.no_grad():
    logits = model_w2v(input_values).logits

# Из логитов берём индекс символа с максимальной вероятностью для каждого временного шага.
# CTC предполагает, что выход — последовательность символов (включая повторения и "пустые" символы).
pred_ids = torch.argmax(logits, dim=-1)

# processor.batch_decode преобразует ID символов в итоговый текст.
# Он удаляет пустые символы, повторы, переводит индексы в реальные буквы и т.п.
text_wav2vec = processor.batch_decode(pred_ids)[0]

print("Wav2Vec2.0 распознавание:", text_wav2vec)
print("Whisper распознавание:", result["text"])

## 4. Подсчёт WER с библиотекой

WER (Word Error Rate) — метрика качества распознавания речи.
Она показывает, на сколько процентов результат распознавания отличается от эталонного текста.

Формула:

WER = (S + D + I) / N

Где:

S (Substitutions) - замены слов

D (Deletions) - пропуски слов

I (Insertions) - лишние вставленные слова

N - количество слов в эталонном тексте

Например, WER = 0.10 → система ошиблась на 10%.

In [ ]:
import jiwer
reference = "Привет, как дела"
hypothesis = "Привет как дела"
error = jiwer.wer(reference, hypothesis)
print(f"WER: {error:.2f}")

## 5. TTS генерация

In [ ]:
import torch
from scipy.io.wavfile import write
import IPython.display as ipd

# Загружаем модель из репозитория NVIDIA через torch.hub.
# Tacotron2 -- модель, которая преобразует текст в мел-спектрограмму.
tacotron2 = torch.hub.load('nvidia/DeepLearningExamples:torchhub', 'nvidia_tacotron2')

# Переводим модель в режим инференса (оценки).
# eval() отключает dropout, переводит BatchNorm в режим "оценки".
# Делается всегда перед генерацией
tacotron2.eval()

# WaveGlow -- нейросетевой вокодер.
# Преобразует мел-спектрограммы в финальный аудиосигнал (волновую форму).
waveglow = torch.hub.load('nvidia/DeepLearningExamples:torchhub', 'nvidia_waveglow')

waveglow.eval()

def tts(text, filename='tts_output.wav'):

    # Преобразуем текст в последовательность ID.
    # text_to_sequence токенизирует текст и преобразует символы в числовые ID
    # english_cleaners -- предобработка текста (удаление лишних символов, нормализация)
    # Далее заворачиваем в LongTensor, добавляем batch размерности [1, N].
    sequences = torch.LongTensor([[tacotron2.text_to_sequence(text, ['english_cleaners'])]]).cuda()

    with torch.no_grad():

        # tacotron2.infer() принимает последовательность ID символов и генерирует мел-спектрограмму:
        #   - mel_outputs: черновая мел-спектрограмма
        #   - mel_outputs_postnet: улучшенная мел-спектрограмма после PostNet
        #   - alignments: карты выравнивания (attention) между текстом и спектрограммой
        mel_outputs, mel_outputs_postnet, _, alignments = tacotron2.infer(sequences)

        # waveglow.infer() преобразует мел-спектрограмму в аудиосигнал (массив PCM данных)
        audio = waveglow.infer(mel_outputs_postnet)

    # Переводим аудио в NumPy (убираем с GPU).
    audio = audio[0].data.cpu().numpy()

    # Сохраняем .wav
    # Частота 22050 Гц -- стандартный sample rate Tacotron2/WaveGlow.
    write(filename, 22050, audio)

    print(f"Файл сохранён: {filename}")

    # Возвращаем объект для воспроизведения в Jupyter/Colab
    return ipd.Audio(filename)

tts("Hello! This is a test of text to speech synthesis.")

## Самостоятельные задания
Попробуйте реализовать небольшие компоненты S2T и T2S самостоятельно.

### Задание 1. CTC decode
Напишите функцию, которая принимает список символов и blank ('_'), и возвращает текст после схлопывания дублей и blank-символов.
**Подсказка:** Используйте цикл по элементам и проверяйте предыдущий символ.

In [ ]:
# TODO: реализовать функцию ctc_decode
def ctc_decode(sequence):
    pass

sequence = ['м','м','_','и','и','и','_','р','_','_']
decoded = ctc_decode(sequence)
print(decoded)

In [ ]:
assert ctc_decode(['м','м','_','и','и','и','_','р','_','_']) == "мир"
assert ctc_decode(['п','р','_','и','в','в','е','т']) == "привет"
assert ctc_decode(['а','а','а','_','_','б','б','в']) == "абв"
assert ctc_decode(['_','_','я','я','_','_']) == "я"
assert ctc_decode([]) == ""

print("Все тесты пройдены!")

### Задание 2. Визуализация спектрограммы с разметкой фонем
Напишите код, который строит мел-спектрограмму аудио. Можно добавить разметку фонем (например, вертикальные линии на оси времени для каждой фонемы). В ячейке ниже приведён пример кода.

In [ ]:
# TODO: реализовать визуализацию спектрограммы с фонемами
import matplotlib.pyplot as plt
import librosa, librosa.display
import numpy as np


plt.colorbar(format='%+2.0f dB')
plt.title('Mel-Spectrogram с разметкой фонем')
plt.show()

In [ ]:
# @title

import matplotlib.pyplot as plt
import librosa, librosa.display
import numpy as np

def plot_mel_with_phonemes(mel, sr, hop_length, phonemes, times):
    """
    mel: np.array [n_mels, time]
    sr: sample rate
    hop_length: шаг STFT (например, 256 или 512)
    phonemes: список строк, напр. ['m', 'i', 'r']
    times: список временных позиций тех же фонем (в секундах или в индексах фреймов)
    """

    plt.figure(figsize=(10, 4))

    # Показываем мел-спектрограмму
    librosa.display.specshow(
        librosa.power_to_db(mel, ref=np.max),
        sr=sr,
        hop_length=hop_length,
        x_axis='time',
        y_axis='mel'
    )

    # Добавляем подписи фонем поверх
    for phoneme, t in zip(phonemes, times):
        plt.text(
            t,                # позиция по x
            mel.shape[0]-1,   # верх спектрограммы
            phoneme,
            color='white',
            fontsize=12,
            ha='center',
            va='top',
            bbox=dict(facecolor='black', alpha=0.6)
        )

    plt.colorbar(format='%+2.0f dB')
    plt.title('Mel-Spectrogram с разметкой фонем')
    plt.tight_layout()
    plt.show()


# Пример вызова
mel = np.abs(librosa.stft(np.random.randn(16000)))[:128]  # Фейковая мел-спектрограмма для демонстрации
phonemes = ['m', 'i', 'r']
times = [0.1, 0.5, 0.8]

plot_mel_with_phonemes(mel, sr=16000, hop_length=256, phonemes=phonemes, times=times)

### Задание 3. Подсчёт WER вручную
Реализуйте простой подсчёт Word Error Rate между эталонным текстом и гипотезой без использования готовых библиотек.

In [ ]:
# TODO: реализовать подсчёт WER вручную
def wer(reference, hypothesis):
    pass

reference = "Привет, как дела"
hypothesis = "Привет как дела"
error = wer(reference, hypothesis)
print(f"WER: {error:.2f}")

In [ ]:
assert abs(wer("hello world", "hello world") - 0.0) < 1e-6
assert abs(wer("hello world", "hello") - 0.5) < 1e-6
assert abs(wer("я люблю чай", "я обожаю чай") - (1/3)) < 1e-6
assert abs(wer("a b c", "x y z") - 1.0) < 1e-6
assert abs(wer("one two three", "") - 1.0) < 1e-6
assert abs(wer("Привет, как дела", "Привет как дела") - (1/3)) < 1e-6

print("Все тесты пройдены!")